# 🏦 05. End-to-End Banking Fraud Operations Simulator
### Project: XAI System in Banking: AI Anomaly-Driven Credit Card Fraud Detection and Interpretation
**Objective:** Interactive demonstration simulating a real-time banking transaction stream. The system evaluates incoming transactions through the dual-stage Anomaly + Deep Learning pipeline, determines actionable banking protocol tiers, and automatically formats Bank Security Operations Alert Cards.

In [1]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import joblib, torch, shap
from IPython.display import Markdown, display

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

from src.data.preprocessing import DataPipeline
from src.models.autoencoder import DeepAutoencoder
from src.xai.analyst_report import BankingAnalystReporter
from src.utils.device import get_device

device = get_device("auto")
print(f"Device: {device}")

/home/swayam-shah/repos/XAI-System-in-Banking-AI-Anomaly-Driven-Credit-Card-Fraud-Detection-and-Interpretation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


## 1. Load Production Artifacts

In [2]:
pipeline = DataPipeline(processed_dir=str(ROOT_DIR / "data/processed"))
data = pipeline.load_processed()

xgb_model = joblib.load(ROOT_DIR / "results/models/xgboost.joblib")
ae_ckpt = torch.load(ROOT_DIR / "results/models/autoencoder.pt", map_location=device, weights_only=False)

autoencoder = DeepAutoencoder(input_dim=data.X_test.shape[1], latent_dim=8).to(device)
autoencoder.load_state_dict(ae_ckpt["model_state_dict"] if "model_state_dict" in ae_ckpt else ae_ckpt)
threshold = ae_ckpt.get("threshold", 1.5048)
reporter = BankingAnalystReporter(output_dir=str(ROOT_DIR / "results/explanations"))
explainer = shap.TreeExplainer(getattr(xgb_model, "model", xgb_model))
print("All production models successfully initialized.")

All production models successfully initialized.


## 2. Simulate Transaction Processing Stream
We simulate incoming transaction evaluation:
1. Fast Autoencoder Reconstruction Check
2. Deep Classifier Probability Scoring
3. Dynamic Protocol Recommendation
4. Instant Analyst Card Compilation

In [3]:
def process_transaction(txn_id: str, x_vector: np.ndarray):
    # 1. Anomaly Check
    with torch.no_grad():
        t_x = torch.as_tensor(x_vector.reshape(1, -1), dtype=torch.float32, device=device)
        recon_err = float(autoencoder.compute_reconstruction_error(t_x, reduction="mean").cpu().numpy()[0])
    
    # 2. Classifier Probability
    prob = float(xgb_model.predict_proba(x_vector.reshape(1, -1))[0, 1])
    
    # 3. SHAP attribution
    shap_vals = explainer(x_vector.reshape(1, -1)).values[0]
    
    card = reporter.generate_card(
        transaction_id=txn_id,
        probability=prob,
        feature_names=data.feature_names,
        feature_values=x_vector,
        shap_values=shap_vals,
        recon_error=recon_err,
        threshold=threshold,
    )
    
    md = reporter.render_markdown(card, filename=f"demo_card_{txn_id}.md")
    return card, md

## 3. Case Study A: Detected Fraud Transaction

In [4]:
fraud_idx = np.where(data.y_test == 1)[0][0]
card_a, md_a = process_transaction("CASE-001-FRAUD", data.X_test[fraud_idx])
display(Markdown(md_a))

[INFO] Saved analyst alert card to: /home/swayam-shah/repos/XAI-System-in-Banking-AI-Anomaly-Driven-Credit-Card-Fraud-Detection-and-Interpretation/results/explanations/demo_card_CASE-001-FRAUD.md


# 🔴 Bank Security Operations Alert Card: Transaction #CASE-001-FRAUD

**Audit Status:** Complete | **Model System:** AI Anomaly-Driven Credit Fraud Engine

---
### Executive Summary
- **Fraud Risk Verdict:** `CRITICAL RISK`
- **Estimated Fraud Probability:** **`99.28%`**
- **Unsupervised Anomaly State:** ⚠️ ANOMALOUS (High Reconstruction Error) (Reconstruction MSE: `3.50708` vs Threshold: `1.50490`)

---
### Root Cause Analysis (XAI Feature Attribution)
#### 🚨 Primary Red Flags (Driving Risk UP):
1. **`V14`** (Normalized Value: `-4.998`) → SHAP Impact: `+5.7184`
2. **`V10`** (Normalized Value: `-4.102`) → SHAP Impact: `+1.1060`
3. **`V16`** (Normalized Value: `-2.991`) → SHAP Impact: `+0.9514`
4. **`V17`** (Normalized Value: `-5.216`) → SHAP Impact: `+0.9385`

#### 🛡️ Mitigating Factors (Driving Risk DOWN):
1. **`V7`** (Normalized Value: `1.318`) → SHAP Impact: `-0.8917`
2. **`Hour_Cos`** (Normalized Value: `-0.475`) → SHAP Impact: `-0.2988`
3. **`V18`** (Normalized Value: `-2.212`) → SHAP Impact: `-0.2372`

---
### Recommended Banking Protocol
> IMMEDIATE ACTION REQUIRED: Block transaction immediately, place temporary hold on credit card account, and dispatch instant SMS/App 2-factor identity verification prompt.

### Regulatory & Compliance Audit Trail
```text
Transaction evaluated by XAI Hybrid Banking Pipeline. Supervised fraud probability: 99.28%. Autoencoder reconstruction residual is 2.33x of the normal transaction boundary. Key risk drivers identified via SHAP Shapley values.
```

## 4. Case Study B: Legitimate Everyday Purchase

In [5]:
normal_idx = np.where(data.y_test == 0)[0][0]
card_b, md_b = process_transaction("CASE-002-NORMAL", data.X_test[normal_idx])
display(Markdown(md_b))

[INFO] Saved analyst alert card to: /home/swayam-shah/repos/XAI-System-in-Banking-AI-Anomaly-Driven-Credit-Card-Fraud-Detection-and-Interpretation/results/explanations/demo_card_CASE-002-NORMAL.md


# 🟢 Bank Security Operations Alert Card: Transaction #CASE-002-NORMAL

**Audit Status:** Complete | **Model System:** AI Anomaly-Driven Credit Fraud Engine

---
### Executive Summary
- **Fraud Risk Verdict:** `LOW RISK (LEGITIMATE)`
- **Estimated Fraud Probability:** **`0.02%`**
- **Unsupervised Anomaly State:** ✅ NORMAL PATTERN (Reconstruction MSE: `0.52021` vs Threshold: `1.50490`)

---
### Root Cause Analysis (XAI Feature Attribution)
#### 🚨 Primary Red Flags (Driving Risk UP):
1. **`V20`** (Normalized Value: `-3.205`) → SHAP Impact: `+0.7294`
2. **`V4`** (Normalized Value: `1.096`) → SHAP Impact: `+0.4544`
3. **`V24`** (Normalized Value: `0.373`) → SHAP Impact: `+0.1323`
4. **`V15`** (Normalized Value: `-0.826`) → SHAP Impact: `+0.0500`

#### 🛡️ Mitigating Factors (Driving Risk DOWN):
1. **`V14`** (Normalized Value: `0.161`) → SHAP Impact: `-1.8770`
2. **`V12`** (Normalized Value: `0.629`) → SHAP Impact: `-1.2377`
3. **`V8`** (Normalized Value: `1.244`) → SHAP Impact: `-0.9556`

---
### Recommended Banking Protocol
> APPROVE: Transaction parameters align with standard legitimate customer behavior.

### Regulatory & Compliance Audit Trail
```text
Transaction evaluated by XAI Hybrid Banking Pipeline. Supervised fraud probability: 0.02%. Autoencoder reconstruction residual is 0.35x of the normal transaction boundary. Key risk drivers identified via SHAP Shapley values.
```